In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('../..'))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from mplsoccer import Pitch
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import KMeans
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

In [ ]:
QUERY = """
-- primary table
CREATE TEMP TABLE player_rel AS SELECT
    player_id,
    match_id,
    team_id,
    player,
    position,
    match_seconds,
    CASE
        WHEN type = 'Complete Carry' THEN end_x
        ELSE x
        END AS x,
    CASE
        WHEN type = 'Complete Carry' THEN end_y
        ELSE y
        END AS y
FROM
    events
WHERE
    (
        (type = 'Pass') OR
        (type = 'Shot') OR
        ((type = 'Complete Carry') AND (carry_end_outcome = 'Other'))
    )
    AND position <> 'Goalkeeper'
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2);

-- secondary table from "player_rel"
CREATE TEMP TABLE player_rel_match_avg AS SELECT
    player_id,
    match_id,
    first(player) AS player,
    first(team_id) AS team_id,
    first(position ORDER BY match_seconds) AS position,
    avg(x) AS avg_rel_x,
    avg(y) AS avg_rel_y,
    avg(40 - ABS(y - 40)) AS avg_normrel_y,
    count(*) AS rel_total
FROM player_rel
GROUP BY player_id, match_id;

-- primary table
CREATE TEMP TABLE player_rec AS SELECT
    player_id,
    match_id,
    x,
    y
FROM
    events
WHERE
    (
        (type = 'Ball Receipt*') OR
        (type = 'Ball Recovery') OR
        ((type = '50/50') AND ("50_50".outcome.name = 'Won')) OR
        ((type = 'Duel') AND (duel_outcome = 'Won')) OR
        ((type = 'Interception') AND (interception_outcome = 'Won')) OR
        ((type = 'Pass') AND (pass_type = 'Recovery')) OR
        ((type = 'Shot') AND (shot_type = 'Recovery'))
    )
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2);

-- secondary table from "player_rec"
CREATE TEMP TABLE player_rec_match_avg AS SELECT
    player_id,
    match_id,
    avg(x) AS avg_rec_x,
    avg(y) AS avg_rec_y,
    avg(40 - ABS(y - 40)) AS avg_normrec_y,
    count(*) AS rec_total
FROM player_rec
GROUP BY player_id, match_id;

-- returned table
SELECT
    a.player_id AS player_id,
    a.player AS player,
    a.match_id AS match_id,
    a.team_id AS team_id,
    a.position AS position,
    a.avg_rel_x AS average_release_x,
    a.avg_rel_y AS average_release_y,
    a.avg_normrel_y AS average_norm_release_y,
    a.rel_total AS rel_total,
    b.avg_rec_x AS average_reception_x,
    b.avg_rec_y AS average_reception_y,
    b.avg_normrec_y AS average_norm_reception_y,
    b.rec_total AS rec_total
FROM player_rel_match_avg a
INNER JOIN player_rec_match_avg b ON
    a.player_id = b.player_id AND
    a.match_id = b.match_id
WHERE
    (a.rel_total >= $1 AND
    b.rec_total >= $2);
"""

In [ ]:
statements = [s.strip() for s in QUERY.split(';') if s.strip()]

setup_statements = statements[:-1]
final_query = statements[-1]

# Indices into setup_statements
# 0: player_rel, 1: player_rel_match_avg, 2: player_rec, 3: player_rec_match_avg
PRIMARY_TABLES = [0, 2]
SECONDARY_TABLES = [1, 3]

DB_PATH = '../../data/events.db'
MATCHWEEK_RANGE = [0, 40]
MIN_COUNTS = [20, 20]

con = duckdb.connect(DB_PATH, read_only=True)
for i in PRIMARY_TABLES:
    con.execute(setup_statements[i], MATCHWEEK_RANGE)
for i in SECONDARY_TABLES:
    con.execute(setup_statements[i])
match_df = con.execute(final_query, MIN_COUNTS).df()
con.close()

In [ ]:
FEATURES = [
    'average_release_x',
    'average_norm_release_y',
    'average_reception_x',
    'average_norm_reception_y',
]
K_RANGE = list(range(2, 11))
RANDOM_STATE = 42

# Drop rows with missing features, then standard scale
cluster_df = match_df.dropna(subset=FEATURES).copy()
X = StandardScaler().fit_transform(cluster_df[FEATURES])

inertias = []
silhouettes = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    # sample_size speeds this up on large data; set to None to use all rows
    silhouettes.append(
        silhouette_score(X, labels, sample_size=10_000, random_state=RANDOM_STATE)
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_RANGE, inertias, marker='o')
axes[0].set_title('Elbow method')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_xticks(K_RANGE)

axes[1].plot(K_RANGE, silhouettes, marker='o', color='tab:orange')
axes[1].set_title('Silhouette score')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Mean silhouette score')
axes[1].set_xticks(K_RANGE)

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
CLUSTER_FEATURES = [
    'average_release_x',
    'average_norm_release_y',
    'average_reception_x',
    'average_norm_reception_y',
]
K_PLOT_RANGE = list(range(3, 8))
RANDOM_STATE = 42
POINT_ALPHA = 0.15      # lower this if the pitches look too busy

cluster_df = match_df.dropna(subset=CLUSTER_FEATURES).copy()
X = StandardScaler().fit_transform(cluster_df[CLUSTER_FEATURES])

pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, axes = pitch.draw(nrows=2, ncols=3, figsize=(20, 11))
axes = axes.ravel()
colors = plt.cm.tab10.colors

for ax, k in zip(axes, K_PLOT_RANGE):
    cluster_df['cluster'] = KMeans(
        n_clusters=k, n_init=10, random_state=RANDOM_STATE
    ).fit_predict(X)

    for c, grp in cluster_df.groupby('cluster'):
        color = colors[c % len(colors)]

        # one line per player_id, match_id pair: reception -> release
        pitch.lines(
            grp['average_reception_x'], grp['average_reception_y'],
            grp['average_release_x'], grp['average_release_y'],
            ax=ax, color=color, lw=0.8, alpha=POINT_ALPHA, zorder=2,
        )
        pitch.scatter(
            grp['average_reception_x'], grp['average_reception_y'],
            ax=ax, marker='o', s=12, color=color, alpha=POINT_ALPHA * 2, zorder=3,
        )
        pitch.scatter(
            grp['average_release_x'], grp['average_release_y'],
            ax=ax, marker='s', s=12, color=color, alpha=POINT_ALPHA * 2, zorder=3,
        )

    ax.set_title(f'k = {k} (n = {len(cluster_df)})')

# Use the spare sixth panel as a legend
axes[-1].axis('off')
axes[-1].legend(
    handles=[
        Line2D([], [], marker='o', linestyle='', color='0.5', markersize=9,
               label='Reception (per player-match)'),
        Line2D([], [], marker='s', linestyle='', color='0.5', markersize=9,
               label='Release (per player-match)'),
        Line2D([], [], color='0.5', linewidth=1, label='Reception → release'),
    ],
    loc='center', fontsize=12, frameon=False,
)

plt.tight_layout()
plt.show()

In [ ]:
CLUSTER_FEATURES = [
    'average_release_x',
    'average_norm_release_y',
    'average_reception_x',
    'average_norm_reception_y',
]
K_RANGE = list(range(2, 11))
RANDOM_STATE = 42
SILHOUETTE_SAMPLE = None   # set to e.g. 10_000 if silhouette is too slow

cluster_df = match_df.dropna(subset=CLUSTER_FEATURES).copy()
X = StandardScaler().fit_transform(cluster_df[CLUSTER_FEATURES])

# Ward linkage on the full scaled data (Euclidean distance)
Z = linkage(X, method='ward')

# Validation metrics for each cut of the tree
rows = []
for k in K_RANGE:
    labels = fcluster(Z, t=k, criterion='maxclust')
    wss = sum(((X[labels == c] - X[labels == c].mean(axis=0)) ** 2).sum()
              for c in np.unique(labels))
    rows.append({
        'k': k,
        'wss': wss,
        'silhouette': silhouette_score(
            X, labels, sample_size=SILHOUETTE_SAMPLE, random_state=RANDOM_STATE
        ),
        'calinski_harabasz': calinski_harabasz_score(X, labels),
        'davies_bouldin': davies_bouldin_score(X, labels),
        'merge_height': Z[-(k - 1), 2],  # distance at which k -> k-1 clusters merge
    })
metrics_df = pd.DataFrame(rows).set_index('k')
print(metrics_df.round(3))

# ---- Dendrogram ----
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=30, show_leaf_counts=True,
           leaf_rotation=90, leaf_font_size=9, ax=ax)
ax.set_title('Ward linkage dendrogram (truncated to last 30 merges)')
ax.set_xlabel('Cluster size (or index)')
ax.set_ylabel('Ward distance')
plt.tight_layout()
plt.show()

# ---- Validation metrics per depth ----
fig, axes = plt.subplots(1, 5, figsize=(24, 4))
panels = [
    ('wss', 'Within-cluster SS (elbow)', 'tab:blue'),
    ('silhouette', 'Silhouette (higher better)', 'tab:orange'),
    ('calinski_harabasz', 'Calinski-Harabasz (higher better)', 'tab:green'),
    ('davies_bouldin', 'Davies-Bouldin (lower better)', 'tab:red'),
    ('merge_height', 'Merge height (big jump = natural cut)', 'tab:purple'),
]
for ax, (col, title, color) in zip(axes, panels):
    ax.plot(metrics_df.index, metrics_df[col], marker='o', color=color)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Number of clusters (k)')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(K_RANGE)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import fcluster

K_PLOT_RANGE = [3, 6]
POINT_ALPHA = 0.2  # Adjust alpha if needed for visibility

pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, axes = pitch.draw(nrows=1, ncols=3, figsize=(18, 6))
axes = axes.ravel()
colors = plt.cm.tab10.colors

for ax, k in zip(axes[:2], K_PLOT_RANGE):
    # Cut the already computed Ward tree (Z) for k clusters
    cluster_df['cluster'] = fcluster(Z, t=k, criterion='maxclust')

    for c, grp in cluster_df.groupby('cluster'):
        color = colors[(c - 1) % len(colors)]  # fcluster labels start at 1

        # One line per player_id, match_id pair: reception -> release
        pitch.lines(
            grp['average_reception_x'], grp['average_reception_y'],
            grp['average_release_x'], grp['average_release_y'],
            ax=ax, color=color, lw=0.8, alpha=POINT_ALPHA, zorder=2,
        )
        pitch.scatter(
            grp['average_reception_x'], grp['average_reception_y'],
            ax=ax, marker='o', s=15, color=color, alpha=POINT_ALPHA * 2, zorder=3,
        )
        pitch.scatter(
            grp['average_release_x'], grp['average_release_y'],
            ax=ax, marker='s', s=15, color=color, alpha=POINT_ALPHA * 2, zorder=3,
        )

    ax.set_title(f'Ward Linkage (k = {k}) | n = {len(cluster_df)}', fontsize=12, fontweight='bold')

# Use the third panel as a legend
axes[2].axis('off')
axes[2].legend(
    handles=[
        Line2D([], [], marker='o', linestyle='', color='0.5', markersize=9,
               label='Reception (per player-match)'),
        Line2D([], [], marker='s', linestyle='', color='0.5', markersize=9,
               label='Release (per player-match)'),
        Line2D([], [], color='0.5', linewidth=1, label='Reception → release'),
    ],
    loc='center', fontsize=12, frameon=False,
)

plt.tight_layout()
plt.show()

In [ ]:
# K-means, k = 3: plot each position with clusters as the colour index
k = 3
position_plot_df = match_df.dropna(subset=CLUSTER_FEATURES).copy()
position_plot_X = StandardScaler().fit_transform(position_plot_df[CLUSTER_FEATURES])
position_plot_df['cluster'] = KMeans(
    n_clusters=k, n_init=10, random_state=RANDOM_STATE
).fit_predict(position_plot_X)
position_plot_df = position_plot_df.dropna(subset=['position'])
positions = sorted(position_plot_df['position'].unique(), key=str)
ncols = 4
nrows = (len(positions) + ncols - 1) // ncols
position_plot_colors = plt.cm.tab10.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, axes = pitch.draw(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = np.asarray(axes).ravel()

for ax, position in zip(axes, positions):
    position_df = position_plot_df[position_plot_df['position'] == position]
    for cluster, group in position_df.groupby('cluster'):
        color = position_plot_colors[cluster % len(position_plot_colors)]
        pitch.lines(
            group['average_reception_x'], group['average_reception_y'],
            group['average_release_x'], group['average_release_y'],
            ax=ax, color=color, lw=0.8, alpha=0.2, zorder=2,
        )
        pitch.scatter(
            group['average_reception_x'], group['average_reception_y'],
            ax=ax, marker='o', s=15, color=color, alpha=0.4, zorder=3,
        )
        pitch.scatter(
            group['average_release_x'], group['average_release_y'],
            ax=ax, marker='s', s=15, color=color, alpha=0.4, zorder=3,
        )
    ax.set_title(f'{position} (n = {len(position_df)})')

for ax in axes[len(positions):]:
    ax.axis('off')

fig.suptitle(f'K-means (k = {k}) by position', fontsize=16)
fig.legend(
    handles=[
        Line2D([], [], color=position_plot_colors[cluster % len(position_plot_colors)], linewidth=2,
               label=f'Cluster {cluster + 1}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    loc='lower center', ncol=k, frameon=False,
)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


In [ ]:
# K-means, k = 4: plot each position with clusters as the colour index
k = 4
position_plot_df = match_df.dropna(subset=CLUSTER_FEATURES).copy()
position_plot_X = StandardScaler().fit_transform(position_plot_df[CLUSTER_FEATURES])
position_plot_df['cluster'] = KMeans(
    n_clusters=k, n_init=10, random_state=RANDOM_STATE
).fit_predict(position_plot_X)
position_plot_df = position_plot_df.dropna(subset=['position'])
positions = sorted(position_plot_df['position'].unique(), key=str)
ncols = 4
nrows = (len(positions) + ncols - 1) // ncols
position_plot_colors = plt.cm.tab10.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, axes = pitch.draw(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = np.asarray(axes).ravel()

for ax, position in zip(axes, positions):
    position_df = position_plot_df[position_plot_df['position'] == position]
    for cluster, group in position_df.groupby('cluster'):
        color = position_plot_colors[cluster % len(position_plot_colors)]
        pitch.lines(
            group['average_reception_x'], group['average_reception_y'],
            group['average_release_x'], group['average_release_y'],
            ax=ax, color=color, lw=0.8, alpha=0.2, zorder=2,
        )
        pitch.scatter(
            group['average_reception_x'], group['average_reception_y'],
            ax=ax, marker='o', s=15, color=color, alpha=0.4, zorder=3,
        )
        pitch.scatter(
            group['average_release_x'], group['average_release_y'],
            ax=ax, marker='s', s=15, color=color, alpha=0.4, zorder=3,
        )
    ax.set_title(f'{position} (n = {len(position_df)})')

for ax in axes[len(positions):]:
    ax.axis('off')

fig.suptitle(f'K-means (k = {k}) by position', fontsize=16)
fig.legend(
    handles=[
        Line2D([], [], color=position_plot_colors[cluster % len(position_plot_colors)], linewidth=2,
               label=f'Cluster {cluster + 1}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    loc='lower center', ncol=k, frameon=False,
)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


In [ ]:
# K-means, k = 5: plot each position with clusters as the colour index
k = 5
position_plot_df = match_df.dropna(subset=CLUSTER_FEATURES).copy()
position_plot_X = StandardScaler().fit_transform(position_plot_df[CLUSTER_FEATURES])
position_plot_df['cluster'] = KMeans(
    n_clusters=k, n_init=10, random_state=RANDOM_STATE
).fit_predict(position_plot_X)
position_plot_df = position_plot_df.dropna(subset=['position'])
positions = sorted(position_plot_df['position'].unique(), key=str)
ncols = 4
nrows = (len(positions) + ncols - 1) // ncols
position_plot_colors = plt.cm.tab10.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, axes = pitch.draw(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = np.asarray(axes).ravel()

for ax, position in zip(axes, positions):
    position_df = position_plot_df[position_plot_df['position'] == position]
    for cluster, group in position_df.groupby('cluster'):
        color = position_plot_colors[cluster % len(position_plot_colors)]
        pitch.lines(
            group['average_reception_x'], group['average_reception_y'],
            group['average_release_x'], group['average_release_y'],
            ax=ax, color=color, lw=0.8, alpha=0.2, zorder=2,
        )
        pitch.scatter(
            group['average_reception_x'], group['average_reception_y'],
            ax=ax, marker='o', s=15, color=color, alpha=0.4, zorder=3,
        )
        pitch.scatter(
            group['average_release_x'], group['average_release_y'],
            ax=ax, marker='s', s=15, color=color, alpha=0.4, zorder=3,
        )
    ax.set_title(f'{position} (n = {len(position_df)})')

for ax in axes[len(positions):]:
    ax.axis('off')

fig.suptitle(f'K-means (k = {k}) by position', fontsize=16)
fig.legend(
    handles=[
        Line2D([], [], color=position_plot_colors[cluster % len(position_plot_colors)], linewidth=2,
               label=f'Cluster {cluster + 1}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    loc='lower center', ncol=k, frameon=False,
)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


In [ ]:
# K-means, k = 6: plot each position with clusters as the colour index
k = 6
position_plot_df = match_df.dropna(subset=CLUSTER_FEATURES).copy()
position_plot_X = StandardScaler().fit_transform(position_plot_df[CLUSTER_FEATURES])
position_plot_df['cluster'] = KMeans(
    n_clusters=k, n_init=10, random_state=RANDOM_STATE
).fit_predict(position_plot_X)
position_plot_df = position_plot_df.dropna(subset=['position'])
positions = sorted(position_plot_df['position'].unique(), key=str)
ncols = 4
nrows = (len(positions) + ncols - 1) // ncols
position_plot_colors = plt.cm.tab10.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, axes = pitch.draw(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = np.asarray(axes).ravel()

for ax, position in zip(axes, positions):
    position_df = position_plot_df[position_plot_df['position'] == position]
    for cluster, group in position_df.groupby('cluster'):
        color = position_plot_colors[cluster % len(position_plot_colors)]
        pitch.lines(
            group['average_reception_x'], group['average_reception_y'],
            group['average_release_x'], group['average_release_y'],
            ax=ax, color=color, lw=0.8, alpha=0.2, zorder=2,
        )
        pitch.scatter(
            group['average_reception_x'], group['average_reception_y'],
            ax=ax, marker='o', s=15, color=color, alpha=0.4, zorder=3,
        )
        pitch.scatter(
            group['average_release_x'], group['average_release_y'],
            ax=ax, marker='s', s=15, color=color, alpha=0.4, zorder=3,
        )
    ax.set_title(f'{position} (n = {len(position_df)})')

for ax in axes[len(positions):]:
    ax.axis('off')

fig.suptitle(f'K-means (k = {k}) by position', fontsize=16)
fig.legend(
    handles=[
        Line2D([], [], color=position_plot_colors[cluster % len(position_plot_colors)], linewidth=2,
               label=f'Cluster {cluster + 1}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    loc='lower center', ncol=k, frameon=False,
)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


In [ ]:
# K-means, k = 7: plot each position with clusters as the colour index
k = 7
position_plot_df = match_df.dropna(subset=CLUSTER_FEATURES).copy()
position_plot_X = StandardScaler().fit_transform(position_plot_df[CLUSTER_FEATURES])
position_plot_df['cluster'] = KMeans(
    n_clusters=k, n_init=10, random_state=RANDOM_STATE
).fit_predict(position_plot_X)
position_plot_df = position_plot_df.dropna(subset=['position'])
positions = sorted(position_plot_df['position'].unique(), key=str)
ncols = 4
nrows = (len(positions) + ncols - 1) // ncols
position_plot_colors = plt.cm.tab10.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, axes = pitch.draw(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = np.asarray(axes).ravel()

for ax, position in zip(axes, positions):
    position_df = position_plot_df[position_plot_df['position'] == position]
    for cluster, group in position_df.groupby('cluster'):
        color = position_plot_colors[cluster % len(position_plot_colors)]
        pitch.lines(
            group['average_reception_x'], group['average_reception_y'],
            group['average_release_x'], group['average_release_y'],
            ax=ax, color=color, lw=0.8, alpha=0.2, zorder=2,
        )
        pitch.scatter(
            group['average_reception_x'], group['average_reception_y'],
            ax=ax, marker='o', s=15, color=color, alpha=0.4, zorder=3,
        )
        pitch.scatter(
            group['average_release_x'], group['average_release_y'],
            ax=ax, marker='s', s=15, color=color, alpha=0.4, zorder=3,
        )
    ax.set_title(f'{position} (n = {len(position_df)})')

for ax in axes[len(positions):]:
    ax.axis('off')

fig.suptitle(f'K-means (k = {k}) by position', fontsize=16)
fig.legend(
    handles=[
        Line2D([], [], color=position_plot_colors[cluster % len(position_plot_colors)], linewidth=2,
               label=f'Cluster {cluster + 1}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    loc='lower center', ncol=k, frameon=False,
)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


In [ ]:
# Ward linkage, k = 3: plot each position with clusters as the colour index
k = 3
position_plot_df = cluster_df.copy()
position_plot_df['cluster'] = fcluster(Z, t=k, criterion='maxclust')
position_plot_df = position_plot_df.dropna(subset=['position'])
positions = sorted(position_plot_df['position'].unique(), key=str)
ncols = 4
nrows = (len(positions) + ncols - 1) // ncols
position_plot_colors = plt.cm.tab10.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, axes = pitch.draw(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = np.asarray(axes).ravel()

for ax, position in zip(axes, positions):
    position_df = position_plot_df[position_plot_df['position'] == position]
    for cluster, group in position_df.groupby('cluster'):
        color = position_plot_colors[(cluster - 1) % len(position_plot_colors)]
        pitch.lines(
            group['average_reception_x'], group['average_reception_y'],
            group['average_release_x'], group['average_release_y'],
            ax=ax, color=color, lw=0.8, alpha=0.2, zorder=2,
        )
        pitch.scatter(
            group['average_reception_x'], group['average_reception_y'],
            ax=ax, marker='o', s=15, color=color, alpha=0.4, zorder=3,
        )
        pitch.scatter(
            group['average_release_x'], group['average_release_y'],
            ax=ax, marker='s', s=15, color=color, alpha=0.4, zorder=3,
        )
    ax.set_title(f'{position} (n = {len(position_df)})')

for ax in axes[len(positions):]:
    ax.axis('off')

fig.suptitle(f'Ward linkage (k = {k}) by position', fontsize=16)
fig.legend(
    handles=[
        Line2D([], [], color=position_plot_colors[(cluster - 1) % len(position_plot_colors)], linewidth=2,
               label=f'Cluster {cluster}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    loc='lower center', ncol=k, frameon=False,
)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


In [ ]:
# Ward linkage, k = 6: plot each position with clusters as the colour index
k = 6
position_plot_df = cluster_df.copy()
position_plot_df['cluster'] = fcluster(Z, t=k, criterion='maxclust')
position_plot_df = position_plot_df.dropna(subset=['position'])
positions = sorted(position_plot_df['position'].unique(), key=str)
ncols = 4
nrows = (len(positions) + ncols - 1) // ncols
position_plot_colors = plt.cm.tab10.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, axes = pitch.draw(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = np.asarray(axes).ravel()

for ax, position in zip(axes, positions):
    position_df = position_plot_df[position_plot_df['position'] == position]
    for cluster, group in position_df.groupby('cluster'):
        color = position_plot_colors[(cluster - 1) % len(position_plot_colors)]
        pitch.lines(
            group['average_reception_x'], group['average_reception_y'],
            group['average_release_x'], group['average_release_y'],
            ax=ax, color=color, lw=0.8, alpha=0.2, zorder=2,
        )
        pitch.scatter(
            group['average_reception_x'], group['average_reception_y'],
            ax=ax, marker='o', s=15, color=color, alpha=0.4, zorder=3,
        )
        pitch.scatter(
            group['average_release_x'], group['average_release_y'],
            ax=ax, marker='s', s=15, color=color, alpha=0.4, zorder=3,
        )
    ax.set_title(f'{position} (n = {len(position_df)})')

for ax in axes[len(positions):]:
    ax.axis('off')

fig.suptitle(f'Ward linkage (k = {k}) by position', fontsize=16)
fig.legend(
    handles=[
        Line2D([], [], color=position_plot_colors[(cluster - 1) % len(position_plot_colors)], linewidth=2,
               label=f'Cluster {cluster}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    loc='lower center', ncol=k, frameon=False,
)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()
